# Mechanisms of Action prediction

A sample can have several mechanism labels. The task is to estimate a separate probability for each of the 206 scored labels using gene expression, cell viability, treatment duration and dose. This notebook evaluates the neural networks used in the original five-model ensemble and one change to its probability estimates.

**Run all cells in order.** On Kaggle, attach the *Mechanisms of Action (MoA) Prediction* competition data. On the group's Windows computer, the notebook recognizes `C:/cs4486/lish-moa/`. Elsewhere, extract the CSVs into `data/` next to this notebook or set `MOA_DATA_DIR` before starting Jupyter. Outputs go to `moa_submission/` locally and `/kaggle/working/` on Kaggle.

[Competition and data](https://www.kaggle.com/competitions/lish-moa/data) · [Evaluation](https://www.kaggle.com/competitions/lish-moa/overview/evaluation)

In [1]:
# Imports and configuration
# Requires numpy, pandas, scikit-learn >= 1.6, and torch.
import os
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")

import random
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.model_selection import GroupKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# An explicit path takes precedence over Kaggle mounts and the local data/ folder.
candidates = [
    Path("/kaggle/input/lish-moa"),
    Path("/kaggle/input/competitions/lish-moa"),
    Path("data"),
    Path("C:/cs4486/lish-moa"),
]
if "MOA_DATA_DIR" in os.environ:
    DATA_DIR = Path(os.environ["MOA_DATA_DIR"]).expanduser()
else:
    DATA_DIR = next((p for p in candidates if (p / "train_features.csv").is_file()), Path("data"))
OUTPUT_DIR = Path(os.environ.get(
    "MOA_OUTPUT_DIR", "/kaggle/working" if Path("/kaggle/input").is_dir() else "moa_submission"
))

# One fixed sensitivity test: shrink each probability 10% toward its training-fold prior.
PRIOR_WEIGHT = 0.10
# Choose "mlp" to reproduce the original submission rule.
SUBMISSION_METHOD = "prior_blend"

SEED = 42
N_MODELS = 5
# Fixed training durations from the completed run in the source notebook.
EPOCHS_PER_MODEL = (4, 2, 10, 4, 3)
HIDDEN_DIMS = (256, 128)
DROPOUT = 0.25
BATCH_SIZE = 256
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-4

torch.set_num_threads(min(4, os.cpu_count() or 1))
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")


import platform
from importlib.metadata import version
print("Python", platform.python_version(), "| device", DEVICE, "| CPU threads", torch.get_num_threads())
print(", ".join(f"{p} {version(p)}" for p in ("torch", "scikit-learn", "pandas", "numpy")))


Python 3.11.0 | device cpu | CPU threads 4
torch 2.15.0, scikit-learn 1.9.1, pandas 2.3.2, numpy 2.3.2


## Data and labels

Match feature rows, target rows and drug IDs by `sig_id`. Use the template to fix the target-column order. The label counts below show how few positive examples some folds can contain.

In [2]:
# Read and align the competition data
required_files = [
    "train_features.csv", "train_targets_scored.csv", "train_drug.csv",
    "test_features.csv", "sample_submission.csv",
]
missing_files = [name for name in required_files if not (DATA_DIR / name).is_file()]
if missing_files:
    raise FileNotFoundError(
        f"Missing competition files in {DATA_DIR}: {missing_files}. "
        "Attach the lish-moa competition data or update DATA_DIR."
    )


def read_features(name):
    columns = pd.read_csv(DATA_DIR / name, nrows=0).columns
    dtypes = {c: np.float32 for c in columns if c.startswith(("g-", "c-"))}
    frame = pd.read_csv(DATA_DIR / name, dtype={"sig_id": str, **dtypes})
    return frame.set_index("sig_id", verify_integrity=True)


train = read_features("train_features.csv")
test = read_features("test_features.csv")
targets = pd.read_csv(
    DATA_DIR / "train_targets_scored.csv", dtype={"sig_id": str}
).set_index("sig_id", verify_integrity=True)
drugs = pd.read_csv(DATA_DIR / "train_drug.csv", dtype=str).set_index(
    "sig_id", verify_integrity=True
)
template = pd.read_csv(DATA_DIR / "sample_submission.csv", dtype={"sig_id": str})
# The 402 unscored labels are optional and are not inputs to this model.

if set(targets.index) != set(train.index) or set(drugs.index) != set(train.index):
    raise ValueError("Training features, targets, and drug IDs must contain the same samples.")
if template.sig_id.isna().any() or template.sig_id.duplicated().any():
    raise ValueError("The submission template must contain unique, nonmissing sample IDs.")
if set(template.sig_id) != set(test.index):
    raise ValueError("The submission template and test features must contain the same samples.")

target_columns = [c for c in template.columns if c != "sig_id"]
if len(target_columns) != 206 or set(target_columns) != set(targets.columns):
    raise ValueError("The competition requires the same 206 scored targets as the template.")

targets = targets.loc[train.index, target_columns]
drugs = drugs.loc[train.index]
numeric_columns = [c for c in train.columns if c.startswith(("g-", "c-"))]
feature_columns = numeric_columns + ["cp_time", "cp_dose"]
X = train[feature_columns]
X_test = test[feature_columns]
y = targets.to_numpy(dtype=np.float32)
groups = drugs.drug_id.to_numpy(dtype=str)
treated_indices = np.flatnonzero(train.cp_type.eq("trt_cp").to_numpy())
test_controls = test.cp_type.eq("ctl_vehicle").to_numpy()


train_controls = train.cp_type.eq("ctl_vehicle").to_numpy()
for frame in (train, test):
    if frame.index.hasnans or not frame.cp_type.isin(["ctl_vehicle", "trt_cp"]).all():
        raise ValueError("Check for missing sample IDs or unexpected treatment types.")
if drugs.drug_id.isna().any() or drugs.loc[~train_controls, "drug_id"].nunique() < N_MODELS:
    raise ValueError("Drug grouping requires complete IDs and at least five treated drugs.")
if not np.isin(y, [0, 1]).all() or y[train_controls].any():
    raise ValueError("Expected binary targets and all-zero labels for vehicle controls.")
print(f"{len(train):,} training samples | {len(test):,} test samples | {len(target_columns)} targets")
print(f"{len(numeric_columns)} measured features | {len(treated_indices):,} treated training samples | "
      f"{len(np.unique(groups[treated_indices])):,} treated drugs")
print(f"{train_controls.sum():,} train controls | {test_controls.sum():,} test controls")
positive_counts = y[treated_indices].sum(axis=0)
print(f"Positive entries among treated samples: {y[treated_indices].mean():.3%} | "
      f"positives per target (min / median / max): "
      f"{positive_counts.min():.0f} / {np.median(positive_counts):.0f} / {positive_counts.max():.0f}")


23,814 training samples | 3,982 test samples | 206 targets
872 measured features | 21,948 treated training samples | 3,288 treated drugs
1,866 train controls | 358 test controls
Positive entries among treated samples: 0.373% | positives per target (min / median / max): 1 / 38 / 832


## Model

Within each fold, impute and standardize the measured features using treated training rows and one-hot encode time and dose. The original MLP has two hidden layers (256 and 128 units), LayerNorm, ReLU and 25% dropout. The final layer produces 206 logits and learns with binary cross-entropy and AdamW. Each output bias starts at the corresponding training-fold log odds.

In [3]:
# Feature preprocessing
def make_preprocessor():
    return ColumnTransformer([
        ("numeric", Pipeline([
            ("imputer", SimpleImputer(strategy="median", keep_empty_features=True)),
            ("scale", StandardScaler()),
        ]), numeric_columns),
        ("conditions", OneHotEncoder(
            categories=[[24, 48, 72], ["D1", "D2"]],
            handle_unknown="error", sparse_output=False,
        ), ["cp_time", "cp_dose"]),
    ], remainder="drop")


def transform_features(preprocessor, frame, fit=False):
    values = preprocessor.fit_transform(frame) if fit else preprocessor.transform(frame)
    return np.ascontiguousarray(values, dtype=np.float32)


In [4]:
# Neural network and training
def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True
    torch.use_deterministic_algorithms(True)


class MoAMLP(nn.Module):
    def __init__(self, input_dim, prior):
        super().__init__()
        layers = []
        for width in HIDDEN_DIMS:
            layers.extend([
                nn.Linear(input_dim, width),
                nn.LayerNorm(width),
                nn.ReLU(),
                nn.Dropout(DROPOUT),
            ])
            input_dim = width
        self.hidden = nn.Sequential(*layers)
        self.output = nn.Linear(input_dim, len(prior))
        initial_prior = np.clip(np.asarray(prior, dtype=np.float64), 1e-6, 1 - 1e-6)
        with torch.no_grad():
            nn.init.normal_(self.output.weight, mean=0.0, std=0.01)
            self.output.bias.copy_(torch.as_tensor(
                np.log(initial_prior / (1 - initial_prior)), dtype=torch.float32
            ))

    def forward(self, values):
        return self.output(self.hidden(values))


def fit_model(features, labels, prior, epochs, seed):
    seed_everything(seed)
    model = MoAMLP(features.shape[1], prior).to(DEVICE)
    optimizer = torch.optim.AdamW(
        model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY
    )
    loader = DataLoader(
        TensorDataset(
            torch.from_numpy(np.ascontiguousarray(features, dtype=np.float32)),
            torch.from_numpy(np.ascontiguousarray(labels, dtype=np.float32)),
        ),
        batch_size=BATCH_SIZE,
        shuffle=True,
        generator=torch.Generator().manual_seed(seed),
        num_workers=0,
        drop_last=False,
    )
    criterion = nn.BCEWithLogitsLoss()
    model.train()
    for _ in range(epochs):
        for batch_x, batch_y in loader:
            batch_x, batch_y = batch_x.to(DEVICE), batch_y.to(DEVICE)
            optimizer.zero_grad(set_to_none=True)
            loss = criterion(model(batch_x), batch_y)
            loss.backward()
            optimizer.step()
    return model


def predict_probabilities(model, features):
    model.eval()
    probabilities = np.empty((len(features), len(target_columns)), dtype=np.float64)
    with torch.inference_mode():
        for start in range(0, len(features), BATCH_SIZE):
            stop = min(start + BATCH_SIZE, len(features))
            batch_x = torch.from_numpy(features[start:stop]).to(DEVICE)
            probabilities[start:stop] = torch.sigmoid(model(batch_x)).cpu().numpy()
    return probabilities


## Validation and probability estimates

Five shuffled folds (seed 42) hold out entire treated drugs. No validation or test sample contributes to preprocessing or to the label priors. Controls have verified zero training labels and receive zero predictions in every method. We distribute them across folds for scoring and also report a treated-only score.

Each validation sample is scored by the one model that held it out. The test submission averages predictions from all five fold models. The original per-fold epoch counts (4, 2, 10, 4, 3) are reused from the group's earlier run and tied to this split. Revisit epoch selection inside each training fold if you change the folds.

| Method | Probability for a treated sample |
|---|---|
| Frequency baseline | Per-label training prevalence, smoothed as (positives + 0.5) / (training rows + 1) |
| Original MLP | Sigmoid of each output logit, with the baseline used for a label that is constant in the training fold |
| Prior blend | 90% MLP probability + 10% frequency baseline, using one fixed weight for all targets |

The blend tests whether pulling predictions slightly toward the fold's training frequencies reduces the penalty for confident errors. The holdout rows, model weights and control rule are identical for this comparison. The metric averages binary log loss over every sample and scored target. It uses the natural logarithm and clips probabilities to the interval [10⁻¹⁵, 1 − 10⁻¹⁵]. Lower values are better.

In [5]:
def mean_log_loss(labels, probabilities):
    p = np.asarray(probabilities, dtype=np.float64)
    if labels.shape != p.shape:
        raise ValueError("Labels and probabilities must have identical shapes.")
    if not np.isfinite(p).all() or ((p < 0) | (p > 1)).any():
        raise ValueError("Probabilities must be finite and in [0, 1].")
    p = np.clip(p, 1e-15, 1 - 1e-15)
    return float(np.where(labels == 1, -np.log(p), -np.log1p(-p)).mean())


def fold_predictions(model, features, control_mask, prior, constant_targets):
    mlp = predict_probabilities(model, features)
    mlp[:, constant_targets] = prior[constant_targets]
    frequency = np.broadcast_to(prior, mlp.shape).copy()
    mlp[control_mask] = 0.0
    frequency[control_mask] = 0.0
    return {
        "frequency": frequency,
        "mlp": mlp,
        "prior_blend": (1 - PRIOR_WEIGHT) * mlp + PRIOR_WEIGHT * frequency,
    }

In [6]:
# Train the five-model ensemble and predict test probabilities
splitter = GroupKFold(n_splits=N_MODELS, shuffle=True, random_state=SEED)
test_probabilities = np.zeros((len(test), len(target_columns)), dtype=np.float64)

# Controls share one ID. Assign them separately for scoring; training uses treated rows only.
control_rows = np.random.default_rng(SEED).permutation(np.flatnonzero(train_controls))
control_folds = np.array_split(control_rows, N_MODELS)
fold_assignment = np.full(len(train), -1, dtype=int)
scores = []
for model_index, (relative_train, relative_valid) in enumerate(
    splitter.split(treated_indices, groups=groups[treated_indices])
):
    fit_indices = treated_indices[relative_train]
    valid_treated = treated_indices[relative_valid]
    if np.intersect1d(groups[fit_indices], groups[valid_treated]).size:
        raise ValueError("A treated drug crossed the validation boundary.")
    valid_indices = np.concatenate([valid_treated, control_folds[model_index]])
    if (fold_assignment[valid_indices] != -1).any():
        raise ValueError("A sample was assigned to multiple validation folds.")
    fold_assignment[valid_indices] = model_index + 1
    fit_labels = y[fit_indices]
    counts = fit_labels.sum(axis=0, dtype=np.float64)
    prior = (counts + 0.5) / (len(fit_indices) + 1)
    constant_targets = (counts == 0) | (counts == len(fit_indices))

    preprocessor = make_preprocessor()
    fit_features = transform_features(preprocessor, X.iloc[fit_indices], fit=True)
    test_features = transform_features(preprocessor, X_test)

    print(f"Training model {model_index + 1}/{N_MODELS}...", flush=True)
    model = fit_model(
        fit_features, fit_labels, prior,
        epochs=EPOCHS_PER_MODEL[model_index], seed=SEED + model_index,
    )
    valid_features = transform_features(preprocessor, X.iloc[valid_indices])
    is_control = train_controls[valid_indices]
    predictions = fold_predictions(model, valid_features, is_control, prior, constant_targets)
    for method, p in predictions.items():
        scores.append({
            "fold": model_index + 1, "method": method, "samples": len(valid_indices),
            "treated": len(valid_treated), "epochs": EPOCHS_PER_MODEL[model_index],
            "all_loss": mean_log_loss(y[valid_indices], p),
            "treated_loss": mean_log_loss(y[valid_treated], p[~is_control]),
        })
    test_preds = fold_predictions(model, test_features, test_controls, prior, constant_targets)
    test_probabilities += test_preds[SUBMISSION_METHOD] / N_MODELS
    del model, preprocessor, fit_features, valid_features, test_features, predictions, test_preds

if (fold_assignment == -1).any():
    raise RuntimeError("Every sample must receive a held-out prediction exactly once.")


Training model 1/5...
Training model 2/5...
Training model 3/5...
Training model 4/5...
Training model 5/5...


In [7]:
fold_scores = pd.DataFrame(scores)
fold_table = fold_scores.pivot(index="fold", columns="method", values="all_loss")
print("Mean binary log loss by validation fold (lower is better)")
print(fold_table[["frequency", "mlp", "prior_blend"]].round(6).to_string())

# Every sample has the same number of targets, so row weighting recovers pooled log loss.
pooled = pd.DataFrame([
    {"method": method,
     "all_samples": np.average(rows.all_loss, weights=rows.samples),
     "treated_only": np.average(rows.treated_loss, weights=rows.treated)}
    for method, rows in fold_scores.groupby("method", sort=False)
]).set_index("method")
print("\nPooled out-of-fold log loss")
print(pooled.round(6).to_string())
difference = pooled.loc["mlp", "all_samples"] - pooled.loc["prior_blend", "all_samples"]
wins = (fold_table["prior_blend"] < fold_table["mlp"]).sum()
print(f"\nMLP minus blend loss: {difference:+.6f}. Blend has lower loss in {wins}/{N_MODELS} folds.")

Mean binary log loss by validation fold (lower is better)
method  frequency       mlp  prior_blend
fold                                    
1        0.020416  0.018855     0.018761
2        0.019466  0.017955     0.017951
3        0.023033  0.024242     0.022431
4        0.020099  0.017567     0.017507
5        0.026689  0.022800     0.022924

Pooled out-of-fold log loss
             all_samples  treated_only
method                                
frequency       0.022193      0.024080
mlp             0.020452      0.022191
prior_blend     0.020094      0.021802

MLP minus blend loss: +0.000359. Blend has lower loss in 4/5 folds.


## Results on held-out drugs

| Method | All-sample log loss | Treated-only log loss |
|---|---:|---:|
| Frequency baseline | 0.022193 | 0.024080 |
| Original MLP | 0.020452 | 0.022191 |
| 90% MLP + 10% training frequency | **0.020094** | **0.021802** |

The fixed blend reduced all-sample log loss by **1.75%** relative to the original MLP. It helped on four of the five folds. Fold 3 accounted for most of the gain (0.024242 to 0.022431), while fold 5 worsened slightly (0.022800 to 0.022924). The treated-only score also improved. These results apply to this five-fold split. An official Kaggle score or a ranking under a shared course split requires separate evaluation.

## Submission and validation records

The submission follows the sample file's row and column order. The additional CSVs record validation fold membership and losses. Kaggle evaluates the submission against hidden test labels. The local comparison measures performance on held-out drugs under this split.

In [8]:
# Write the required submission file
if not np.isfinite(test_probabilities).all():
    raise FloatingPointError("The model produced nonfinite probabilities; submission was not written.")
predictions = pd.DataFrame(
    np.clip(test_probabilities, 0.0, 1.0), index=test.index, columns=target_columns
)
submission = template.copy()
submission[target_columns] = predictions.loc[template.sig_id, target_columns].to_numpy()
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
if not submission.columns.equals(template.columns) or not submission.sig_id.equals(template.sig_id):
    raise ValueError("Submission layout differs from the official template.")
if ((test_probabilities < 0) | (test_probabilities > 1)).any() or test_probabilities[test_controls].any():
    raise ValueError("Unexpected test probabilities or nonzero control predictions.")
fold_scores.to_csv(OUTPUT_DIR / "fold_scores.csv", index=False)
pooled.to_csv(OUTPUT_DIR / "cv_summary.csv")
pd.DataFrame({"sig_id": train.index, "fold": fold_assignment}).to_csv(
    OUTPUT_DIR / "fold_assignments.csv", index=False
)
submission_path = OUTPUT_DIR / "submission.csv"
submission.to_csv(submission_path, index=False, float_format="%.12g")
print(f"Wrote {submission_path.name} | {SUBMISSION_METHOD} | {len(submission):,} rows x {len(submission.columns)} columns")


Wrote submission.csv | prior_blend | 3,982 rows x 207 columns


### Contribution note

The supplied submission notebook contains the MLP and its training pipeline. This revision used OpenAI Codex to diagnose the missing validation output, add scoring and the training-frequency comparison, and verify the saved submission. The group should identify its own design contributions and any earlier assistance in its course report.